In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
import tensorflow_hub as hub

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {tf.keras.__version__}")

TensorFlow Version: 2.21.0
Keras Version: 3.14.1


In [3]:
tfds.list_builders() # check for the avilable datasets inside tensor_flow

['abstract_reasoning',
 'accentdb',
 'aeslc',
 'aflw2k3d',
 'ag_news_subset',
 'ai2_arc',
 'ai2_arc_with_ir',
 'ai2dcaption',
 'aloha_mobile',
 'amazon_us_reviews',
 'anli',
 'answer_equivalence',
 'arc',
 'asimov_dilemmas_auto_val',
 'asimov_dilemmas_scifi_train',
 'asimov_dilemmas_scifi_val',
 'asimov_injury_val',
 'asimov_multimodal_auto_val',
 'asimov_multimodal_manual_val',
 'asimov_v2_constraints_with_rationale',
 'asimov_v2_constraints_without_rationale',
 'asimov_v2_injuries',
 'asimov_v2_videos',
 'asqa',
 'asset',
 'assin2',
 'asu_table_top_converted_externally_to_rlds',
 'austin_buds_dataset_converted_externally_to_rlds',
 'austin_sailor_dataset_converted_externally_to_rlds',
 'austin_sirius_dataset_converted_externally_to_rlds',
 'bair_robot_pushing_small',
 'bc_z',
 'bccd',
 'beans',
 'bee_dataset',
 'beir',
 'berkeley_autolab_ur5',
 'berkeley_cable_routing',
 'berkeley_fanuc_manipulation',
 'berkeley_gnm_cory_hall',
 'berkeley_gnm_recon',
 'berkeley_gnm_sac_son',
 'berkel

In [4]:
(train_data, validation_data, test_data), metadata = tfds.load(
'imdb_reviews',
split=['train[0%:80%]', 'train[80%:90%]', 'train[90%:100%]'],
with_info=True,
as_supervised=True,)  # create 03 datasets, for training, testing and validation

In [5]:
train_data

<_PrefetchDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>

In [6]:
train_examples_batch, train_labels_batch = next(iter(train_data.batch(2))) #read user-reviews
train_examples_batch

I0000 00:00:1780342237.332213 11101157 tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
W0000 00:00:1780342237.333585 11100990 cache_dataset_ops.cc:912] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


<tf.Tensor: shape=(2,), dtype=string, numpy=
array([b"This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it.",
       b'I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on this occasion I fell a

In [7]:
train_labels_batch

<tf.Tensor: shape=(2,), dtype=int64, numpy=array([0, 0])>

In [8]:
pretrained_model = "https://tfhub.dev/google/tf2-preview/gnews-swivel-20dim/1"

# Custom Keras 3 Compatibility Wrapper for Legacy TF-Hub Layers
class HubEmbeddingLayer(tf.keras.layers.Layer):
    def __init__(self, handle, trainable=True, **kwargs):
        super().__init__(**kwargs)
        self.handle = handle
        self.trainable = trainable
        self.hub_layer = hub.KerasLayer(handle, trainable=trainable)
        
    def call(self, inputs):
        # Flatten inputs to a 1D string tensor as expected by the Swivel model
        flat_inputs = tf.reshape(inputs, [-1])
        return self.hub_layer(flat_inputs)
        
    def compute_output_shape(self, input_shape):
        return (input_shape[0], 20)

# Create modern TF-Hub layer
hub_layer = HubEmbeddingLayer(pretrained_model, trainable=True)

In [9]:
hub_layer(train_examples_batch[:2]) # encoded view of the frist two review feedbacks

<tf.Tensor: shape=(2, 20), dtype=float32, numpy=
array([[ 1.765786  , -3.882232  ,  3.9134233 , -1.5557289 , -3.3362343 ,
        -1.7357955 , -1.9954445 ,  1.2989551 ,  5.081598  , -1.1041286 ,
        -2.0503852 , -0.72675157, -0.65675956,  0.24436149, -3.7208383 ,
         2.0954835 ,  2.2969332 , -2.0689783 , -2.9489717 , -1.1315987 ],
       [ 1.8804485 , -2.5852382 ,  3.4066997 ,  1.0982676 , -4.056685  ,
        -4.891284  , -2.785554  ,  1.3874227 ,  3.8476458 , -0.9256538 ,
        -1.896706  ,  1.2113281 ,  0.11474707,  0.76209456, -4.8791065 ,
         2.906149  ,  4.7087674 , -2.3652055 , -3.5015898 , -1.6390051 ]],
      dtype=float32)>

In [10]:
train_examples_batch[:2]

<tf.Tensor: shape=(2,), dtype=string, numpy=
array([b"This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it.",
       b'I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on this occasion I fell a

In [11]:
model = tf.keras.Sequential()
model.add(tf.keras.Input(shape=(), dtype=tf.string)) # Input specifier for Keras 3 symbolic tracing
model.add(hub_layer) # input layer is the encoded output from tensor_flow hub neural networ
model.add(tf.keras.layers.Dense(16,activation="relu"))
model.add(tf.keras.layers.Dense(1,activation="sigmoid"))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hub_embedding_layer             │ (None, 20)             │             0 │
│ (HubEmbeddingLayer)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 353 (1.38 KB)

 Trainable params: 353 (1.38 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])  # back_prop configuration

In [13]:
model.fit(train_data.shuffle(10000).batch(512),
         epochs=20,
         validation_data=validation_data.batch(512),
         verbose=1)

Epoch 1/20



 1/40 ━━━━━━━━━━━━━━━━━━━━ 12s 318ms/step - accuracy: 0.5059 - loss: 1.7550


 6/40 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4992 - loss: 1.6795  


12/40 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4987 - loss: 1.5808


18/40 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5002 - loss: 1.4921

...


19/40 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6879 - loss: 0.5806


25/40 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6875 - loss: 0.5818


31/40 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6876 - loss: 0.5827


37/40 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6873 - loss: 0.5837


40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6867 - loss: 0.5884 - val_accuracy: 0.6940 - val_loss: 0.5813


In [14]:
model.predict(tf.constant(["This is the worst movie I have ever seen",
              "An excellent movie that I enjoyed a lot",
              "how can one make such a horrible movie? there is no story, acting direction everything is very poor"]))



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step


array([[0.38093248],
       [0.6340815 ],
       [0.43646735]], dtype=float32)

In [15]:
model.predict(tf.constant(["An excellent movie that I enjoyed a lot"]))



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step


array([[0.6340815]], dtype=float32)